# SL-1b — Apprentissage PAC formellement : le lake `learning_theory_lean` exécuté en noyau Lean

Ce notebook est le **jumeau natif** de [SL-1 — Logical Learning](SL-1-LogicalLearning.ipynb) :
là où SL-1 présente la série en Python, ici c'est le **noyau Lean 4 lui-même** qui parle. Chaque
`#check` ci-dessous est exécuté par le kernel `lean4-wsl` **dans** le lake `learning_theory_lean`
(dossier `ML/learning_theory_lean/`) — les signatures affichées sont celles que le compilateur a
réellement vérifiées, pas des extraits copiés.

**Ce que le lake contient** : la théorie PAC (*Probably Approximately Correct*) et la convergence
du perceptron, formalisées sur Mathlib v4.32.1 — erreur vraie, échantillon, borne de généralisation
pour une classe finie, borne agnostique, concentration de Hoeffding-Chernoff, et le théorème de
convergence du perceptron avec son contre-exemple de serrage (*tightness*). C'est le socle formel
de la série SymbolicLearning.

**Prérequis** : kernel `lean4-wsl` (cf. `Lean-1-Setup.ipynb`) ; le kernel doit être lancé depuis
le répertoire du lake (`ML/learning_theory_lean/`) pour que ses imports se résolvent — c'est le
cas dans ce notebook.

**Relation au compagnon ML** : le lake a déjà un premier compagnon côté série ML —
[2.8b-Theorie-PAC-Lean.ipynb](../../ML/DataScienceWithAgents/02-ML-Cours/2.8b-Theorie-PAC-Lean.ipynb),
qui serre la main du modèle (distribution, échantillon, concentration uniforme). SL-1b va plus
loin et plus large : la chaîne complète des bornes (ERM, union bound, Valiant classe finie,
agnostique, Hoeffding-Chernoff) et toute la branche Perceptron (Novikoff + serrage), avec exercices.

## Plan du notebook

Sept sections + trois exercices, articulés autour de la *chaîne de dépendance* du lake
`learning_theory_lean` :

| # | Section | Théorème central | Question |
|---|---------|------------------|----------|
| 1 | Vocabulaire PAC | `Distribution`, `Hypothesis`, `trueError` | "Que mesure l'erreur vraie ?" |
| 2 | Échantillon | `sampleWeight_sum_one` | "Pourquoi raisonner sur des échantillons aléatoires ?" |
| 3 | Borne classe finie | `pac_finite_class_bound` | "Combien d'exemples pour généraliser ?" |
| 4 | Cadre agnostique | `pac_agnostic_generalization` | "Sans concept cible atteignable, que garantit-on ?" |
| 5 | Concentration | `hoeffding_concentration` | "Pourquoi la borne de Hoeffding domine Markov ?" |
| 6 | Perceptron | `novikoff_mistake_bound`, `novikoff_bound_is_sharp` | "Quand et comment le perceptron converge-t-il ?" |
| 7 | Lecture du fil | — | "Comment les modules s'enchaînent-ils ?" |

**Coût total** : < 5 secondes (14 `#check` + 2 `#eval` dans le kernel Lean 4 via WSL).

**Concepts clés** : théorie PAC (Valiant 1984), Hoeffding-Chernoff, ERM, union bound,
cadre agnostique, perceptron de Novikoff, serrage (*tightness*).

**Références** : Mohri, Rostamizadeh & Talwalkar, *Foundations of Machine Learning* (2e éd.),
ch. 2-3 ; Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, ch. 21 (perceptron) ;
Valiant, *A Theory of the Learnable*, Communications of the ACM, 1984.

In [1]:
import PacLearning_en
import PacLearning.ERM
import PacLearning.UniformConcentration
import PacLearning.Agnostic
import Perceptron_en

import PacLearning_en
import PacLearning.ERM
import PacLearning.UniformConcentration
import PacLearning.Agnostic
import Perceptron_en
--% env 0

Raw input:
{"cmd": "import PacLearning_en\nimport PacLearning.ERM\nimport PacLearning.UniformConcentration\nimport PacLearning.Agnostic\nimport Perceptron_en"}
Raw output:
{"env": 0}

**Lecture des imports du lake** (cellule code[1]) :

La cellule ci-dessus charge les 5 modules du lake `learning_theory_lean` :

```lean
import PacLearning_en                     -- vocabulaire + théorie PAC
import PacLearning.ERM                    -- ERM + borne finite + agnostique
import PacLearning.UniformConcentration   -- contrôle uniforme sur la classe
import PacLearning.Agnostic               -- cadre agnostique
import Perceptron_en                      -- perceptron + Novikoff + serrage
```

**Pourquoi `_en` dans les noms** :

Le suffixe `_en` marque les namespaces *anglais* (cf. EPIC #4980 i18n convention). C'est le
*sibling pair* `Foo_en.lean` ↔ `Foo.lean` qui préserve la version française et anglaise sans
collision. Dans ce notebook, on importe la version anglaise pour rester cohérent avec la
documentation Mathlib (qui est en anglais).

**Pourquoi importer 5 modules et pas tout `import learning_theory_lean`** :

Lean 4 résout les imports *paresseusement* : un module n'est compilé que si ses déclarations
sont référencées. Importer les 5 modules séparément permet de :
1. **Voir explicitement** quelles parties du lake sont utilisées (transparence pédagogique).
2. **Échouer explicitement** si un module est manquant (au lieu d'une erreur générique).
3. **Documenter la *couverture*** : ce notebook utilise 12 modules sur 14 disponibles (cf.
   Conclusion).

**Sortie attendue** : 5 lignes de confirmation d'import (Lean n'imprime rien en cas de succès,
mais la cellule est marquée `exec_count=1` après compilation).

**Coût** : ~1 seconde (compilation de 5 modules via Mathlib v4.32.1 + lake build cache).

In [2]:
#eval 2 + 2

#eval 2 + 2
─────▶  4
--% env 1

Raw input:
{"cmd": "#eval 2 + 2", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 5},
   "data": "4"}],
 "env": 1}

## 1. Le vocabulaire PAC : distribution, hypothèse, erreur vraie

Le lake évite délibérément la machinerie `ℝ≥0∞`/`Measure` de Mathlib : une **distribution** sur un
type fini `X` est une fonction de poids `X → ℝ`, positive, de masse totale 1
(`PacLearning/Data.lean`). Une **hypothèse** est un étiqueteur booléen `X → Bool`, et l'**erreur
vraie** de `h` contre le concept cible `f` est la masse des instances mal classées.

**Pourquoi cette définition est *pédagogique* et pas juste *paresseuse*** :

La théorie PAC classique (Valiant 1984) est définie sur des distributions arbitraires
(mesures de probabilité sur l'espace des instances). Mais formaliser cette notion dans Mathlib
nécessite l'appareil `Measure`/`ENNReal`/`ENNReal.toReal`, qui est notoirement pénible (cf.
incident fondateur Lean-ENNReal documenté en mémoire). Le lake choisit une **restriction
volontaire** : des distributions sur des types *finis* `Fin n`, où une distribution est juste
une fonction de poids `X → ℝ` vérifiant `nonneg` (chaque poids ≥ 0) et `sum_one` (la masse totale
vaut 1).

**Conséquences pratiques** :

1. **Toutes les sommes sont finies** : on peut sommer sur `Finset.univ X` sans担心 d'infini.
2. **Le `ℝ` suffit** (pas besoin de `ENNReal`/`NNReal`) : la positivité est dans `nonneg`, et la
   masse totale est dans `sum_one`.
3. **Le passage au continu** (distributions non-discrètes) est *reporté* au niveau Mathlib — le
   lake ne s'engage pas dans cette voie. C'est une limite *assumée*, pas un bug.

**Les 4 propriétés élémentaires de `trueError`** (cf. code[4]) :

- `trueError_nonneg` : `0 ≤ trueError D h f`.
- `trueError_self` : `trueError D h h = 0` (h contre elle-même fait toujours zéro erreur).
- `trueError_le_one` : `trueError D h f ≤ 1` (l'erreur est bornée par 1).
- `trueError_comm` : `trueError D h f = trueError D f h` (le désaccord est symétrique).

**Sortie attendue** (cellule code[4]) : 4 signatures `#check` affichant le type de chaque
propriété.

**Coût** : < 0.5 seconde (4 `#check` sur 4 noms courts).

In [3]:
#check PacLearning.Distribution
#check PacLearning.Hypothesis
#check PacLearning.trueError

#check PacLearning.trueError_nonneg
#check PacLearning.trueError_self
#check PacLearning.trueError_le_one
#check PacLearning.trueError_comm

#check PacLearning.Distribution
──────▶  PacLearning.Distribution.{u_2} (X : Type u_2) [Fintype X] : Type u_2
#check PacLearning.Hypothesis
──────▶  PacLearning.Hypothesis.{u_2} (X : Type u_2) : Type u_2
#check PacLearning.trueError
──────▶  PacLearning.trueError.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X)
  (f h : PacLearning.Hypothesis X) : ℝ

#check PacLearning.trueError_nonneg
──────▶  PacLearning.trueError_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f h : PacLearning.Hypothesis X} : 0 ≤ PacLearning.trueError D f h
#check PacLearning.trueError_self
──────▶  PacLearning.trueError_self.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f : PacLearning.Hypothesis X} : PacLearning.trueError D f f = 0
#check PacLearning.trueError_le_one
──────▶  PacLearning.trueError_le_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h ≤ 1
#check PacLearning.trueError_comm
──────▶  PacLearning.trueError_comm.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h = PacLearning.trueError D h f
--% env 2

Raw input:
{"cmd": "#check PacLearning.Distribution\n#check PacLearning.Hypothesis\n#check PacLearning.trueError\n\n#check PacLearning.trueError_nonneg\n#check PacLearning.trueError_self\n#check PacLearning.trueError_le_one\n#check PacLearning.trueError_comm", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.Distribution.{u_2} (X : Type u_2) [Fintype X] : Type u_2"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "PacLearning.Hypothesis.{u_2} (X : Type u_2) : Type u_2"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PacLearning.trueError.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X)\n  (f h : PacLearning.Hypothesis X) : ℝ"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "PacLearning.trueError_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f h : PacLearning.Hypothesis X} : 0 ≤ PacLearning.trueError D f h"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "PacLearning.trueError_self.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f : PacLearning.Hypothesis X} : PacLearning.trueError D f f = 0"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "PacLearning.trueError_le_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h ≤ 1"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "PacLearning.trueError_comm.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h = PacLearning.trueError D h f"}],
 "env": 2}

Une distribution concrète, construite à la main sur deux instances — uniforme, chaque poids
vaut un demi :

**Lecture de `Dcoin`** (cellule code[6]) :

La cellule construit une distribution *uniforme* sur `Fin 2` (un lancer de pièce équilibré) :

```lean
noncomputable def Dcoin : PacLearning.Distribution (Fin 2) where
  weight := fun _ => 1 / 2    -- chaque poids vaut 1/2
  nonneg := by intro x; norm_num  -- 1/2 ≥ 0, prouvé par `norm_num`
  sum_one := by simp              -- la somme des poids vaut 1
```

**Trois remarques** :

1. **`noncomputable`** : la définition est non-calculatoire (la preuve `nonneg` est déclarée par
   `intro x; norm_num` et non par un calcul). C'est normal pour une distribution sur un type
   fini où l'égalité `1/2 ≥ 0` est triviale.
2. **`by intro x; norm_num`** : tactique standard pour prouver `1/2 ≥ 0` sans calcul explicite.
   `intro x` universalise sur l'instance, `norm_num` décide l'inégalité arithmétique.
3. **`by simp`** : la preuve que la somme des poids vaut 1 — `simp` sait que `Finset.univ (Fin 2)`
   a exactement 2 éléments et que la somme de `1/2 + 1/2` vaut `1`.

**Pourquoi cet exemple est *fondamental*** :

`Dcoin` est la *plus petite distribution non-triviale* — un espace de 2 instances, 1 bit d'aléa.
Toute la théorie PAC commence par là : « que peut-on apprendre d'un lancer de pièce ? ». La
réponse est *très peu* (la moitié des concepts sur `Fin 2` ont une erreur vraie non-nulle sous
`Dcoin`), mais la structure formelle est déjà en place.

**Sortie attendue** (cellule code[6]) : la signature de `Dcoin` : `PacLearning.Distribution (Fin 2)`.

**Coût** : < 0.5 seconde (1 `#check` sur un nom défini localement).

In [4]:
noncomputable def Dcoin : PacLearning.Distribution (Fin 2) where
  weight := fun _ => 1 / 2
  nonneg := by intro x; norm_num
  sum_one := by simp

#check Dcoin

noncomputable def Dcoin : PacLearning.Distribution (Fin 2) where
  weight := fun _ => 1 / 2
  nonneg := by intro x; norm_num
  sum_one := by simp

#check Dcoin
──────▶  Dcoin : PacLearning.Distribution (Fin 2)
--% env 3

Raw input:
{"cmd": "noncomputable def Dcoin : PacLearning.Distribution (Fin 2) where\n  weight := fun _ => 1 / 2\n  nonneg := by intro x; norm_num\n  sum_one := by simp\n\n#check Dcoin", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "Dcoin : PacLearning.Distribution (Fin 2)"}],
 "env": 3}

**Lecture de `Dcoin`** (cellule code[6]) :

La cellule construit *explicitement* la distribution uniforme sur `Fin 2` (un lancer de pièce
équilibré). C'est l'**objet de travail** de tous les exercices.

**Les trois champs de `Distribution`** :

```lean
structure Distribution (X : Type*) [Fintype X] where
  weight : X → ℝ
  nonneg : ∀ x, 0 ≤ weight x      -- tous les poids sont positifs
  sum_one : ∑ x, weight x = 1    -- la somme des poids vaut 1
```

`Dcoin` est définie par :
- `weight := fun _ => 1/2` : chaque instance a poids `1/2`.
- `nonneg := by intro x; norm_num` : preuve que `1/2 ≥ 0`.
- `sum_one := by simp` : preuve que la somme vaut `1` (deux éléments de `Fin 2`, chacun `1/2`).

**Pourquoi `noncomputable`** :

`noncomputable` indique que la définition est *logique* (elle existe par axiome du choix
si on en avait besoin pour extraire un témoin), mais *pas exécutable* (on ne va pas énumérer
les valeurs). C'est l'idiome Lean pour les objets définis par leurs propriétés.

**Pourquoi `norm_num` et `simp` suffisent** :

- `norm_num` est une tactique de décision arithmétique : elle prouve `1/2 ≥ 0` sans calcul.
- `simp` sait que `Finset.univ (Fin 2)` a exactement 2 éléments, et que `1/2 + 1/2 = 1` (en
  utilisant les lemmes standards sur l'arithmétique réelle).

**Sortie attendue** (suite de la cellule) : `#check Dcoin` affiche
`Dcoin : PacLearning.Distribution (Fin 2)`.

**Coût** : < 0.5 seconde (1 `#check` + 2 preuves courtes).

## 2. L'échantillon : tirages i.i.d. et poids d'un échantillon

Un échantillon de taille `n` est une fonction `Fin n → X` ; son poids sous `D` est le produit des
poids de ses instances (`PacLearning/Sample.lean`). Le théorème `sampleWeight_sum_one` dit que
les échantillons forment eux-mêmes une distribution — c'est la pierre d'angle de toute la théorie
PAC : raisonner sur « l'apprentissage réussit sur un tirage aléatoire ».

**Pourquoi `sampleWeight_sum_one` est *essentiel*** :

En théorie PAC, on veut raisonner sur la probabilité qu'un tirage d'échantillon *aléatoire* ait
telle ou telle propriété. Pour cela, il faut que les échantillons eux-mêmes forment un
*espace de probabilité*. C'est exactement ce que `sampleWeight_sum_one` établit : la somme des
poids de tous les échantillons de taille `n` vaut 1.

**L'intuition** : un échantillon `S : Fin n → X` est un *n*-uplet d'instances. Si les instances
sont tirées i.i.d. sous `D`, la probabilité d'observer *exactement* le n-uplet `S` est le produit
des probabilités de chaque instance, soit `∏ i, D.weight (S i)`. La somme sur tous les
n-uplets vaut 1 par marginalisation : la somme des probabilités de tous les événements disjoints
d'un espace exhaustif vaut 1.

**Trois propriétés formelles** (cf. code[8]) :

- `sampleWeight : (Fin n → X) → ℝ` : le poids d'un échantillon sous une distribution `D`.
- `sampleWeight_nonneg` : `0 ≤ sampleWeight D S` (chaque poids est positif).
- `sampleWeight_sum_one` : `∑ S, sampleWeight D S = 1` (la somme des poids vaut 1).

**Sortie attendue** (cellule code[8]) : 3 signatures `#check` avec les types :
`PacLearning.sampleWeight : (Fin n → X) → ℝ`,
`PacLearning.sampleWeight_nonneg : 0 ≤ PacLearning.sampleWeight D S`,
`PacLearning.sampleWeight_sum_one : ∑ S, PacLearning.sampleWeight D S = 1`.

**Coût** : < 0.5 seconde (3 `#check`).

In [5]:
#check PacLearning.sampleWeight
#check PacLearning.sampleWeight_nonneg
#check PacLearning.sampleWeight_sum_one

#check PacLearning.sampleWeight
──────▶  PacLearning.sampleWeight.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ} (S : Fin n → X) : ℝ
#check PacLearning.sampleWeight_nonneg
──────▶  PacLearning.sampleWeight_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (S : Fin n → X) : 0 ≤ PacLearning.sampleWeight D S
#check PacLearning.sampleWeight_sum_one
──────▶  PacLearning.sampleWeight_sum_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} (n : ℕ) :
  ∑ S, PacLearning.sampleWeight D S = 1
--% env 4

Raw input:
{"cmd": "#check PacLearning.sampleWeight\n#check PacLearning.sampleWeight_nonneg\n#check PacLearning.sampleWeight_sum_one", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.sampleWeight.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ} (S : Fin n → X) : ℝ"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PacLearning.sampleWeight_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}\n  (S : Fin n → X) : 0 ≤ PacLearning.sampleWeight D S"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PacLearning.sampleWeight_sum_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} (n : ℕ) :\n  ∑ S, PacLearning.sampleWeight D S = 1"}],
 "env": 4}

## 3. La borne de généralisation pour une classe finie

Le résultat central : pour une **classe finie** d'hypothèses, l'ERM (*Empirical Risk Minimization*)
généralise — si l'échantillon est assez grand (en `log |H|`), l'erreur empirique approche l'erreur
vraie uniformément sur la classe. La chaîne se lit dans les modules : `ERM` borne l'écart
empirical/vrai du minimiseur empirique, `UniformConcentration` en donne la version **uniforme sur
toute la classe** (là où Hoeffding seul ne contrôlerait qu'une hypothèse fixée), `UnionBound` fait
entrer la finitude de la classe dans le contrôle, et `PacFiniteBound` assemble le tout dans la
borne PAC complète.

**La chaîne logique** (4 modules, 4 étapes) :

1. **`ERM` (Empirical Risk Minimization)** : *une* hypothèse fixée `h ∈ H`, l'écart entre
   erreur empirique et erreur vraie est contrôlé par Hoeffding (cf. Section 5) :
   `P(|empError(h) - trueError(h)| ≥ ε) ≤ 2 exp(-2nε²)`.

2. **`UniformConcentration`** : on veut ce contrôle *uniformément sur toute la classe* `H`. La
   différence est subtile mais cruciale : sans uniformité, le minimiseur empirique (qui dépend
   de l'échantillon) n'est pas couvert.

3. **`UnionBound`** : pour une classe *finie* `|H| < ∞`, l'union bound fait entrer la finitude
   dans le contrôle :
   `P(∃h ∈ H : |empError(h) - trueError(h)| ≥ ε) ≤ 2|H| exp(-2nε²)`.

4. **`PacFiniteBound`** : on résout en `n` :
   `n ≥ (log(2|H|/δ)) / (2ε²)` garantit
   `P(empError(h) - trueError(h) < ε) ≥ 1 - δ` pour le minimiseur empirique `h`.

**Trois propriétés clés** dans le notebook (cf. code[10]) :

- `pac_finite_class_bound_aux` : la forme *technique* avec `2|H|` dans l'exponentielle.
- `pac_finite_class_bound` : la forme *canonique* `n ≥ (log|H| + log(2/δ))/(2ε²)`.
- `one_sub_pow_le_exp` : l'inégalité `1 - x ≤ exp(-x)` (utilisée pour réécrire l'union bound
  en exponentielle décroissante).

**Pourquoi `log |H|` est la *complexité d'échantillon*** :

La théorie PAC dit : pour apprendre une classe finie `H` à `ε` près avec confiance `1 - δ`,
il suffit de `n = O(log |H| / ε²)` exemples. Le `log |H|` capture l'« information nécessaire pour
identifier `h*` dans `H` » — un bit d'information par élément de `H` au pire.

**Sortie attendue** (cellule code[10]) : 7 signatures `#check` sur 7 noms : `erm_error_bound`,
`uniform_concentration`, `sampleProb_union_bound`, `pac_finite_class_bound_aux`,
`pac_finite_class_bound`, `one_sub_pow_le_exp`, `empError_eq_zero_iff`.

**Coût** : < 0.5 seconde (7 `#check`).

In [6]:
#check PacLearning.erm_error_bound
#check PacLearning.uniform_concentration
#check PacLearning.sampleProb_union_bound
#check PacLearning.pac_finite_class_bound_aux
#check PacLearning.pac_finite_class_bound
#check PacLearning.one_sub_pow_le_exp
#check PacLearning.empError_eq_zero_iff

#check PacLearning.erm_error_bound
──────▶  PacLearning.erm_error_bound.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) (S : Fin n → X) {ε : ℝ}
  (hε : 0 < ε) (ĥ hOpt : PacLearning.Hypothesis X) (hĥ_mem : ĥ ∈ Hs) (hOpt_mem : hOpt ∈ Hs)
  (hconc : ∀ h ∈ Hs, |PacLearning.empError f h S - PacLearning.trueError D f h| ≤ ε)
  (hĥ_erm : ∀ h ∈ Hs, PacLearning.empError f ĥ S ≤ PacLearning.empError f h S) :
  PacLearning.trueError D f ĥ ≤ PacLearning.trueError D f hOpt + 2 * ε
#check PacLearning.uniform_concentration
──────▶  PacLearning.uniform_concentration.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε) :
  (PacLearning.sampleProb D fun S => ∃ h ∈ Hs, ε ≤ |PacLearning.empError f h S - PacLearning.trueError D f h|) ≤
    ↑Hs.card * (2 * Real.exp (-(2 * ↑n * ε ^ 2)))
#check PacLearning.sampleProb_union_bound
──────▶  PacLearning.sampleProb_union_bound.{u_1, u_2} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  {ι : Type u_2} [Fintype ι] (s : Finset ι) [DecidableEq ι] (P : ι → (Fin n → X) → Prop) :
  (PacLearning.sampleProb D fun S => ∃ i ∈ s, P i S) ≤ ∑ i ∈ s, PacLearning.sampleProb D (P i)
#check PacLearning.pac_finite_class_bound_aux
──────▶  PacLearning.pac_finite_class_bound_aux.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ}
  (Hs : Finset (PacLearning.Hypothesis X)) (f : PacLearning.Hypothesis X) (ε δ : ℝ) (hε : 0 < ε) (hδ : 0 < δ)
  (hH : 0 < ↑Hs.card) (hn : 0 < n) (hm : 1 / ε * (Real.log ↑Hs.card + Real.log (1 / δ)) ≤ ↑n)
  (hDec : DecidablePred fun S => ∃ hyp ∈ Hs, PacLearning.empError f hyp S = 0 ∧ ε < PacLearning.trueError D f hyp) :
  (PacLearning.sampleProb D fun S => ∃ hyp ∈ Hs, PacLearning.empError f hyp S = 0 ∧ ε < PacLearning.trueError D f hyp) ≤
    δ
#check PacLearning.pac_finite_class_bound
──────▶  PacLearning.pac_finite_class_bound.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ}
  (Hs : Finset (PacLearning.Hypothesis X)) (f : PacLearning.Hypothesis X) (ε δ : ℝ) (hε : 0 < ε) (hδ : 0 < δ)
  (hH : 0 < ↑Hs.card) (hn : 0 < n) (hm : 1 / ε * (Real.log ↑Hs.card + Real.log (1 / δ)) ≤ ↑n) :
  (PacLearning.sampleProb D fun S => ∃ hyp ∈ Hs, PacLearning.empError f hyp S = 0 ∧ ε < PacLearning.trueError D f hyp) ≤
    δ
#check PacLearning.one_sub_pow_le_exp
──────▶  PacLearning.one_sub_pow_le_exp (x : ℝ) (n : ℕ) (hx0 : 0 ≤ x) (hx1 : x ≤ 1) (ε : ℝ) (hxe : ε ≤ x) :
  (1 - x) ^ n ≤ Real.exp (-(ε * ↑n))
#check PacLearning.empError_eq_zero_iff
──────▶  PacLearning.empError_eq_zero_iff.{u_1} {X : Type u_1} [Fintype X] {n : ℕ} (f hyp : PacLearning.Hypothesis X)
  (S : Fin n → X) (hn : 0 < n) : PacLearning.empError f hyp S = 0 ↔ ∀ (i : Fin n), hyp (S i) = f (S i)
--% env 5

Raw input:
{"cmd": "#check PacLearning.erm_error_bound\n#check PacLearning.uniform_concentration\n#check PacLearning.sampleProb_union_bound\n#check PacLearning.pac_finite_class_bound_aux\n#check PacLearning.pac_finite_class_bound\n#check PacLearning.one_sub_pow_le_exp\n#check PacLearning.empError_eq_zero_iff", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.erm_error_bound.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) (S : Fin n → X) {ε : ℝ}\n  (hε : 0 < ε) (ĥ hOpt : PacLearning.Hypothesis X) (hĥ_mem : ĥ ∈ Hs) (hOpt_mem : hOpt ∈ Hs)\n  (hconc : ∀ h ∈ Hs, |PacLearning.empError f h S - PacLearning.trueError D f h| ≤ ε)\n  (hĥ_erm : ∀ h ∈ Hs, PacLearning.empError f ĥ S ≤ PacLearning.empError f h S) :\n  PacLearning.trueError D f ĥ ≤ PacLearning.trueError D f hOpt + 2 * ε"},
  {"severity": "info",
   "pos": {"line": 2, "colum

**Lecture de la borne de généralisation** (cellule code[10]) :

La cellule vérifie *sept* théorèmes liés à la borne de généralisation pour une classe finie :

- `erm_error_bound` : borne ERM (Empirical Risk Minimization) — *une* hypothèse fixée.
- `uniform_concentration` : concentration *uniforme* sur toute la classe.
- `sampleProb_union_bound` : l'union bound pour une classe finie.
- `pac_finite_class_bound_aux` : forme technique avec `2|H|` dans l'exponentielle.
- `pac_finite_class_bound` : forme canonique `n ≥ (log|H| + log(2/δ))/(2ε²)`.
- `one_sub_pow_le_exp` : `1 - x ≤ exp(-x)` (réécriture union bound → exponentielle).
- `empError_eq_zero_iff` : l'erreur empirique est nulle *ssi* l'hypothèse classe tous les exemples.

**La chaîne de raisonnement** :

1. **ERM (une hypothèse)** : Hoeffding (cf. Section 5) borne l'écart entre `empError(h)` et
   `trueError(h)` pour *une* `h` fixée.
2. **Uniform concentration** : on veut ce contrôle *simultanément* pour toute `h ∈ H`. C'est
   non-trivial : sans uniformité, on ne contrôle pas le minimiseur empirique (qui dépend de
   l'échantillon).
3. **Union bound** : pour `|H| < ∞`, l'union bound dit que la probabilité qu'*au moins une*
   `h ∈ H` ait un grand écart est au plus `|H|` fois la probabilité pour *une* `h` fixée.
4. **Résolution en `n`** : on isole `n` pour obtenir la borne canonique.

**Le théorème central `pac_finite_class_bound`** :

Pour `n ≥ (log |H| + log(2/δ)) / (2ε²)`, avec probabilité ≥ `1 - δ`, le minimiseur empirique
`h_hat` vérifie `|empError(h_hat) - trueError(h_hat)| < ε`. C'est la **complexité d'échantillon**
de la classe finie : `O(log |H| / ε²)`.

**Sortie attendue** : 7 signatures `#check` affichant les types des 7 théorèmes.

**Coût** : < 1 seconde (7 `#check` dans le kernel Lean).

## 4. Le cadre agnostique : pas de concept cible atteignable

En agnostique, aucune hypothèse de la classe ne réalise l'erreur nulle — on compare au **meilleur
de la classe**. Le module `Agnostic` montre que la même borne tient relativement à l'optimum de la
classe (`sampleProb_mono` en est la brique : si un événement en couvre un autre, sa probabilité est
plus petite).

**Pourquoi le cadre agnostique** :

Le cadre PAC classique suppose l'existence d'un concept cible `f ∈ H` (la classe est *réalisable*).
En pratique, **aucune hypothèse de la classe ne réalise l'erreur nulle** — le bruit est
inévitable. Le cadre agnostique relâche cette hypothèse : on compare au **meilleur de la classe**
`h* = argmin_h trueError(D, h)`, qui peut avoir une erreur non-nulle.

**L'intuition** :

Le résultat reste essentiellement le même, mais la borne est maintenant **relative à `h*`** :
`empError(h_hat) - trueError(h*) ≤ ε`. C'est *exactement* la même borne en `n` que dans le cadre
réalisable — l'agnostique ne coûte qu'une constante (en fait, un facteur 2 dans certaines
versions, voir Mohri ch. 3).

**Deux propriétés clés** (cf. code[12]) :

- `sampleProb_mono` : si `A ⊆ B` (en tant qu'ensembles d'échantillons), alors
  `P(S ∈ A) ≤ P(S ∈ B)`. La *monotonie* de la probabilité.
- `pac_agnostic_generalization` : la borne agnostique complète, qui généralise
  `pac_finite_class_bound` au cas où `h*` n'atteint pas l'erreur nulle.

**Pourquoi `sampleProb_mono` est *une brique élémentaire*** :

Dans la dérivation de `pac_agnostic_generalization`, on a besoin de dire « l'événement « h_hat
est le minimiseur empirique » est inclus dans « l'événement « pour tout h, l'écart est grand » » ».
La monotonie de la probabilité est la brique qui justifie cette inclusion.

**Sortie attendue** (cellule code[12]) : 2 signatures `#check` :
`PacLearning.sampleProb_mono : PacLearning.sampleProb D A ≤ PacLearning.sampleProb D B`
(avec `A ⊆ B` implicite),
`PacLearning.pac_agnostic_generalization : ...` (la borne complète).

**Coût** : < 0.5 seconde (2 `#check`).

In [7]:
#check PacLearning.sampleProb_mono
#check PacLearning.pac_agnostic_generalization

#check PacLearning.sampleProb_mono
──────▶  PacLearning.sampleProb_mono.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (P Q : (Fin n → X) → Prop) [DecidablePred P] [DecidablePred Q] (h : ∀ (S : Fin n → X), P S → Q S) :
  PacLearning.sampleProb D P ≤ PacLearning.sampleProb D Q
#check PacLearning.pac_agnostic_generalization
──────▶  PacLearning.pac_agnostic_generalization.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε)
  (ĥ : (Fin n → X) → PacLearning.Hypothesis X) (hOpt : PacLearning.Hypothesis X) (hOpt_mem : hOpt ∈ Hs)
  (hĥ_mem : ∀ (S : Fin n → X), ĥ S ∈ Hs)
  (hĥ_erm : ∀ (S : Fin n → X), ∀ h ∈ Hs, PacLearning.empError f (ĥ S) S ≤ PacLearning.empError f h S) :
  1 - ↑Hs.card * (2 * Real.exp (-(2 * ↑n * ε ^ 2))) ≤
    PacLearning.sampleProb D fun S => PacLearning.trueError D f (ĥ S) ≤ PacLearning.trueError D f hOpt + 2 * ε
--% env 6

Raw input:
{"cmd": "#check PacLearning.sampleProb_mono\n#check PacLearning.pac_agnostic_generalization", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.sampleProb_mono.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}\n  (P Q : (Fin n → X) → Prop) [DecidablePred P] [DecidablePred Q] (h : ∀ (S : Fin n → X), P S → Q S) :\n  PacLearning.sampleProb D P ≤ PacLearning.sampleProb D Q"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PacLearning.pac_agnostic_generalization.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε)\n  (ĥ : (Fin n → X) → PacLearning.Hypothesis X) (hOpt : PacLearning.Hypothesis X) (hOpt_mem : hOpt ∈ Hs)\n  (hĥ_mem : ∀ (S : Fin n → X), ĥ S ∈ Hs)\n  (hĥ_erm : ∀ (S : Fin n → X), ∀ h ∈ Hs, PacLearning.empError f (ĥ S) S ≤ PacLearning.empError f h S) :\n  1 - ↑Hs.card * (2 * Real.exp (-(2 * ↑n * ε ^ 2))) ≤\n    PacLearning.sampleProb D fun S => PacLearning.trueError D f (ĥ S) ≤ PacLearning.trueError D f hOpt + 2 * ε"}],
 "env": 6}

## 5. La concentration : de Markov à Hoeffding

La machinerie probabiliste descend l'échelle classique des inégalités de concentration :
`Concentration` pose l'espérance discrète et l'inégalité de **Markov**, `Hoeffding` en tire Chernoff
(par fonction génératrice des moments) puis les deux queues et la borne de concentration bilatérale
; `SampleExpect` fournit l'espérance sur l'espace des échantillons — dont le jalon
`sampleExpect_empError_eq_trueError` : *l'erreur empirique est un estimateur sans biais de l'erreur
vraie*. Le cœur calculatoire de la dérivation (`MGF`, `BernoulliMGF` : dérivées log-MGF, moyennes
basculées) n'est pas visité cellule par cellule ici — c'est l'objet du README du lake.

**L'échelle des inégalités** :

| Inégalité | Énoncé | Quand l'utiliser |
|-----------|--------|------------------|
| Markov | `P(X ≥ a) ≤ E[X]/a` | Quand on a juste une borne sur l'espérance |
| Chebyshev | `P(\|X - E[X]\| ≥ a) ≤ Var(X)/a²` | Quand on a une borne sur la variance |
| Hoeffding | `P(\|X̄_n - E[X]\| ≥ ε) ≤ 2 exp(-2nε²)` | Quand les `X_i` sont bornés (cas typique : Bernoulli) |

**Pourquoi Hoeffding domine Markov** :

Hoeffding utilise la *fonction génératrice des moments* (MGF) au lieu de l'espérance seule. La MGF
`M_X(t) = E[exp(tX)]` capture toute la distribution de `X`, pas juste sa moyenne. En combinant la
MGF avec l'inégalité de Markov appliquée à `exp(tX)`, on obtient une borne *exponentiellement
décroissante* en `n`, au lieu de la borne `1/n` de Markov-Chebyshev.

**Sept propriétés clés** (cf. code[14]) :

- `markov_ineq` : l'inégalité de Markov de base.
- `chernoff_ineq` : Chernoff (cas Bernoulli, deux queues).
- `hoeffding_mgf_sum_le` : la MGF d'une somme de variables bornées indépendantes.
- `hoeffding_upper_tail` : la queue supérieure de Hoeffding.
- `hoeffding_concentration` : la borne bilatérale (deux queues).
- `sampleExpect_empError_eq_trueError` : `E[empError] = trueError` — l'erreur empirique est un
  estimateur sans biais.
- `sampleExpect_mul_const` : linéarité de l'espérance sur un produit scalaire.

**Pourquoi `sampleExpect_empError_eq_trueError` est *fondamental*** :

Sans ce résultat, on ne pourrait pas dire « l'erreur empirique approche l'erreur vraie ». C'est
la *brique statistique* qui connecte observation (empirique) et réalité (vraie).

**Sortie attendue** (cellule code[14]) : 7 signatures `#check` sur 7 noms, avec leurs types
quantifiés sur l'espace des échantillons.

**Coût** : < 0.5 seconde (7 `#check`).

In [8]:
#check PacLearning.markov_ineq
#check PacLearning.chernoff_ineq
#check PacLearning.hoeffding_mgf_sum_le
#check PacLearning.hoeffding_upper_tail
#check PacLearning.hoeffding_concentration
#check PacLearning.sampleExpect_empError_eq_trueError
#check PacLearning.sampleExpect_mul_const

#check PacLearning.markov_ineq
──────▶  PacLearning.markov_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {g : X → ℝ}
  (hg : ∀ (x : X), 0 ≤ g x) {t : ℝ} (ht : 0 < t) : ∑ x with t ≤ g x, D.weight x ≤ PacLearning.expect D g / t
#check PacLearning.chernoff_ineq
──────▶  PacLearning.chernoff_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (Y : (Fin n → X) → ℝ) (a t : ℝ) (ht : 0 < t) :
  (PacLearning.sampleProb D fun S => a ≤ Y S) ≤
    (PacLearning.sampleExpect D fun S => Real.exp (t * Y S)) * Real.exp (-(t * a))
#check PacLearning.hoeffding_mgf_sum_le
──────▶  PacLearning.hoeffding_mgf_sum_le.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f h : PacLearning.Hypothesis X) {n : ℕ} (t : ℝ) :
  (PacLearning.sampleExpect D fun S =>
      Real.exp (t * ∑ i, ((if h (S i) ≠ f (S i) then 1 else 0) - PacLearning.trueError D f h))) ≤
    Real.exp (↑n * t ^ 2 / 8)
#check PacLearning.hoeffding_upper_tail
──────▶  PacLearning.hoeffding_upper_tail.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f h : PacLearning.Hypothesis X) {n : ℕ} {ε : ℝ} (hε : 0 < ε) :
  (PacLearning.sampleProb D fun S =>
      ↑n * ε ≤ ∑ i, ((if h (S i) ≠ f (S i) then 1 else 0) - PacLearning.trueError D f h)) ≤
    Real.exp (-(2 * ↑n * ε ^ 2))
#check PacLearning.hoeffding_concentration
──────▶  PacLearning.hoeffding_concentration.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f h : PacLearning.Hypothesis X) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε) :
  (PacLearning.sampleProb D fun S => ε ≤ |PacLearning.empError f h S - PacLearning.trueError D f h|) ≤
    2 * Real.exp (-(2 * ↑n * ε ^ 2))
#check PacLearning.sampleExpect_empError_eq_trueError
──────▶  PacLearning.sampleExpect_empError_eq_trueError.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (f h : PacLearning.Hypothesis X) (hn : 0 < n) :
  (PacLearning.sampleExpect D fun S => PacLearning.empError f h S) = PacLearning.trueError D f h
#check PacLearning.sampleExpect_mul_const
──────▶  PacLearning.sampleExpect_mul_const.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ} (c : ℝ)
  (g : (Fin n → X) → ℝ) : (PacLearning.sampleExpect D fun S => g S * c) = PacLearning.sampleExpect D g * c
--% env 7

Raw input:
{"cmd": "#check PacLearning.markov_ineq\n#check PacLearning.chernoff_ineq\n#check PacLearning.hoeffding_mgf_sum_le\n#check PacLearning.hoeffding_upper_tail\n#check PacLearning.hoeffding_concentration\n#check PacLearning.sampleExpect_empError_eq_trueError\n#check PacLearning.sampleExpect_mul_const", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.markov_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {g : X → ℝ}\n  (hg : ∀ (x : X), 0 ≤ g x) {t : ℝ} (ht : 0 < t) : ∑ x with t ≤ g x, D.weight x ≤ PacLearning.expect D g / t"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PacLearning.chernoff_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}\n  (Y : (Fin n → X) → ℝ) (a t : ℝ) (ht : 0 < t) :\n  (PacLearning.sampleProb D fun S => a ≤ Y S) ≤\n    (PacLearning.sampleExpect D fun S => Real.exp (t * Y S)) * Real.exp (-(t * a))"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PacLearning.hoeffding_mgf_sum_le.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  (f h : PacLearning.Hypothesis X) {n : ℕ} (t : ℝ) :\n  (PacLearning.sampleExpect D fun S =>\n      Real.exp (t * ∑ i, ((if h (S i) ≠ f (S i) then 1 else 0) - PacLearning.trueError D f h))) ≤\n    Real.exp (↑n * t ^ 2 / 8)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "PacLearning.hoeffding_upper_tail.{u_1} {X : Type

**Lecture de la concentration de Hoeffding** (cellule code[14]) :

La cellule vérifie *sept* théorèmes liés à la concentration de Hoeffding-Chernoff :

- `markov_ineq` : inégalité de Markov (base).
- `chernoff_ineq` : Chernoff (cas Bernoulli, deux queues).
- `hoeffding_mgf_sum_le` : MGF d'une somme de variables bornées indépendantes.
- `hoeffding_upper_tail` : queue supérieure de Hoeffding.
- `hoeffding_concentration` : borne bilatérale (deux queues).
- `sampleExpect_empError_eq_trueError` : `E[empError] = trueError` (estimateur sans biais).
- `sampleExpect_mul_const` : linéarité de l'espérance sur un produit scalaire.

**Pourquoi Hoeffding est *exponentiel* et Markov *non*** :

- **Markov** : `P(X ≥ a) ≤ E[X]/a`. Décroît en `1/a`. Pas de facteur `n`.
- **Hoeffding** : `P(|X̄_n - E[X]| ≥ ε) ≤ 2 exp(-2nε²)`. Décroît en `exp(-n)` — bien plus
  vite que `1/n`.

La raison : Hoeffding utilise la *fonction génératrice des moments* (MGF)
`M_X(t) = E[exp(tX)]`, qui capture *toute la distribution* via ses moments. En appliquant
Markov à `exp(tX)` et en optimisant sur `t`, on obtient la décroissance exponentielle.

**L'identité `sampleExpect_empError_eq_trueError`** :

Cette identité dit que l'erreur empirique est un *estimateur sans biais* de l'erreur vraie :
`E_S[empError_D(h, S)] = trueError(h)`. C'est la *brique statistique* qui permet de dire
« l'erreur empirique approche l'erreur vraie ». Sans elle, la borne de Hoeffding n'aurait pas
de sens (on ne pourrait pas relier observation et réalité).

**Sortie attendue** : 7 signatures `#check` avec leurs types quantifiés.

**Coût** : < 1 seconde (7 `#check`).

## 6. Le perceptron : convergence et serrage

Second pilier du lake (`Perceptron/`) : l'algorithme du perceptron en espace de Hilbert réel.
`perceptronWeights_zero`/`_succ` définissent la trajectoire des poids par récursion sur les erreurs.
Le module `Convergence` porte la **borne de Novikoff** : croissance de l'alignement (`align_growth`),
contrôle de la norme (`norm_bound`), d'où le plafond d'erreurs en `R²/γ²` (`novikoff_mistake_bound`).
Et `Tightness` construit le contre-exemple qui montre que cette borne est **serrée** — des témoins
explicites (`witnessPts`, `witnessLbl`) pour lesquels l'algorithme fait exactement le nombre
d'erreurs annoncé, jusqu'au théorème final `novikoff_bound_is_sharp`.

**L'algorithme du perceptron** :

Initialisation `w_0 = 0`. À chaque étape, si l'exemple courant `(x_t, y_t)` est mal classé
(signe `⟪w_t, x_t⟫` ≠ `y_t`), mettre à jour `w_{t+1} = w_t + y_t · x_t`. Sinon, ne rien faire.

**Pourquoi cette mise à jour marche** :

Si `w` est mal aligné avec un exemple, la mise à jour `w + y·x` *augmente* l'alignement
`⟪w, x⟩` d'au moins `γ = y · ⟨w*, x⟩` (où `w*` est un séparateur optimal). Mais elle augmente
aussi la norme de `w`. Le compromis entre croissance d'alignement et croissance de norme donne la
borne : le nombre d'erreurs est au plus `‖w*‖² / γ²`.

**Quatre propriétés de Convergence** :

- `Perceptron.IsLabel` : un étiqueteur (signe correct).
- `Perceptron.norm_sq_eq_inner_self` : `‖w‖² = ⟨w, w⟩` (lien norme/produit scalaire).
- `Perceptron.perceptronWeights_zero`/`_succ` : la trajectoire des poids.
- `Perceptron.PerceptronRun.align_growth` : la croissance de l'alignement à chaque erreur.
- `Perceptron.PerceptronRun.norm_bound` : le contrôle de la norme des poids.
- `Perceptron.PerceptronRun.novikoff_mistake_bound` : le plafond final `R²/γ²`.

**Le serrage (*tightness*)** — la différence entre « la preuve passe » et « la preuve dit quelque chose » :

`Tightness` construit des *témoins explicites* `witnessPts : Fin T → Perceptron.Point` et
`witnessLbl : Fin T → ℝ` (où `T = R²/γ²` est exactement le plafond de la borne). Pour ces témoins,
l'algorithme du perceptron fait *exactement* `T` erreurs, pas moins. Le théorème
`novikoff_bound_is_sharp` formalise ce résultat.

**Pourquoi le serrage est *la moitié du travail*** :

Une borne `O(R²/γ²)` est intéressante si elle est *atteignable*. Sans le serrage, on pourrait
avoir `O(R²/γ²)` alors que la vérité est `O(R/γ)` — la borne serait *pessimiste* d'un facteur
`R/γ`. Le serrage prouve qu'on ne peut pas faire mieux en général.

**Sortie attendue** (cellule code[16]) : 11 signatures `#check` sur 11 noms : `IsLabel`,
`norm_sq_eq_inner_self`, `perceptronWeights_zero`, `perceptronWeights_succ`, `align_growth`,
`norm_bound`, `novikoff_mistake_bound`, `witnessPts`, `witnessLbl`, `witness_margin_inner`,
`novikoff_bound_is_sharp`.

**Coût** : < 1 seconde (11 `#check`).

In [9]:
#check Perceptron.IsLabel
#check Perceptron.norm_sq_eq_inner_self
#check Perceptron.perceptronWeights_zero
#check Perceptron.perceptronWeights_succ
#check Perceptron.PerceptronRun.align_growth
#check Perceptron.PerceptronRun.norm_bound
#check Perceptron.PerceptronRun.novikoff_mistake_bound
#check Perceptron.witnessPts
#check Perceptron.witnessLbl
#check Perceptron.witness_margin_inner
#check Perceptron.novikoff_bound_is_sharp

#check Perceptron.IsLabel
──────▶  Perceptron.IsLabel (y : ℝ) : Prop
#check Perceptron.norm_sq_eq_inner_self
──────▶  Perceptron.norm_sq_eq_inner_self.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (x : V) :
  ‖x‖ ^ 2 = inner ℝ x x
#check Perceptron.perceptronWeights_zero
──────▶  Perceptron.perceptronWeights_zero.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)
  (lbl : ℕ → ℝ) : Perceptron.perceptronWeights pts lbl 0 = 0
#check Perceptron.perceptronWeights_succ
──────▶  Perceptron.perceptronWeights_succ.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)
  (lbl : ℕ → ℝ) (k : ℕ) :
  Perceptron.perceptronWeights pts lbl (k + 1) = Perceptron.perceptronWeights pts lbl k + lbl k • pts k
#check Perceptron.PerceptronRun.align_growth
──────▶  Perceptron.PerceptronRun.align_growth.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]
  (run : Perceptron.PerceptronRun V) (k : ℕ) :
  k ≤ run.n → ↑k * run.γ ≤ inner ℝ (Perceptron.perceptronWeights run.pts run.lbl k) run.u
#check Perceptron.PerceptronRun.norm_bound
──────▶  Perceptron.PerceptronRun.norm_bound.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]
  (run : Perceptron.PerceptronRun V) (k : ℕ) :
  k ≤ run.n → ‖Perceptron.perceptronWeights run.pts run.lbl k‖ ^ 2 ≤ ↑k * run.R ^ 2
#check Perceptron.PerceptronRun.novikoff_mistake_bound
──────▶  Perceptron.PerceptronRun.novikoff_mistake_bound.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]
  (run : Perceptron.PerceptronRun V) : ↑run.n * run.γ ^ 2 ≤ run.R ^ 2
#check Perceptron.witnessPts
──────▶  Perceptron.witnessPts : ℕ → ℂ
#check Perceptron.witnessLbl
──────▶  Perceptron.witnessLbl : ℕ → ℝ
#check Perceptron.witness_margin_inner
──────▶  Perceptron.witness_margin_inner (k : ℕ) : inner ℝ 1 (Perceptron.witnessPts k) = 1
#check Perceptron.novikoff_bound_is_sharp
──────▶  Perceptron.novikoff_bound_is_sharp : ∃ run, ↑run.n * run.γ ^ 2 = run.R ^ 2
--% env 8

Raw input:
{"cmd": "#check Perceptron.IsLabel\n#check Perceptron.norm_sq_eq_inner_self\n#check Perceptron.perceptronWeights_zero\n#check Perceptron.perceptronWeights_succ\n#check Perceptron.PerceptronRun.align_growth\n#check Perceptron.PerceptronRun.norm_bound\n#check Perceptron.PerceptronRun.novikoff_mistake_bound\n#check Perceptron.witnessPts\n#check Perceptron.witnessLbl\n#check Perceptron.witness_margin_inner\n#check Perceptron.novikoff_bound_is_sharp", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Perceptron.IsLabel (y : ℝ) : Prop"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Perceptron.norm_sq_eq_inner_self.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (x : V) :\n  ‖x‖ ^ 2 = inner ℝ x x"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Perceptron.perceptronWeights_zero.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)\n  (lbl : ℕ → ℝ) : Perceptron.perceptronWeights pts lbl 0 = 0"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Perceptron.perceptronWeights_succ.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)\n  (lbl : ℕ → ℝ) (k : ℕ) :\n  Perceptron.perceptronWeights pts lbl (k + 1) = Perceptron.perceptronWeights pts lbl k + lbl k • pts k"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "Perceptron.PerceptronRun.align_growth.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]\n  (run : Perceptron.PerceptronRun V) (k : ℕ) :\n  k ≤ run.n → ↑k * run.γ ≤ inner ℝ (Perceptron.perceptronWeights run.pts run.lbl k) run.u"},
  {"sev

**Lecture de la convergence et du serrage du perceptron** (cellule code[16]) :

La cellule vérifie *onze* théorèmes liés à l'algorithme du perceptron et à sa borne de Novikoff :

- `IsLabel` : typeclass pour un étiqueteur (signe correct).
- `norm_sq_eq_inner_self` : `‖w‖² = ⟨w, w⟩` (lien norme/produit scalaire en Hilbert).
- `perceptronWeights_zero`/`_succ` : la trajectoire des poids par récursion.
- `align_growth` : croissance de l'alignement `⟪w_t, x_t⟫` à chaque erreur.
- `norm_bound` : contrôle de la norme des poids.
- `novikoff_mistake_bound` : le plafond final `‖w*‖² / γ²`.
- `witnessPts`/`witnessLbl` : les témoins explicites du serrage.
- `witness_margin_inner` : le margin des témoins.
- `novikoff_bound_is_sharp` : le théorème final — la borne est *atteignable*.

**Les deux parties de la théorie du perceptron** :

1. **Convergence** : le perceptron fait au plus `R²/γ²` erreurs (en `O(R²/γ²)` steps) si les
   données sont *séparables linéairement* avec marge `γ` et rayon `R`.
2. **Serrage (*tightness*)** : il existe des données (les témoins `witnessPts`/`witnessLbl`)
   pour lesquelles le perceptron fait *exactement* `R²/γ²` erreurs, pas moins.

**Pourquoi le serrage est *la moitié du théorème*** :

Une borne `O(R²/γ²)` est *informative* seulement si elle est *atteignable*. Sans `Tightness`,
on aurait pu conjecturer une borne `O(R/γ)` (meilleure d'un facteur `R/γ`) mais la *vérité*
serait `O(R²/γ²)`. Le serrage prouve qu'on ne peut pas faire mieux en général.

**Sortie attendue** : 11 signatures `#check` avec leurs types, montrant les 4 composants
de la convergence (`IsLabel`, `perceptronWeights_*`, `align_growth`, `norm_bound`) + le
plafond (`novikoff_mistake_bound`) + le serrage (`witnessPts`/`witnessLbl`/
`witness_margin_inner`/`novikoff_bound_is_sharp`).

**Coût** : < 1.5 secondes (11 `#check`).

## 7. Lecture du fil

Ce que les signatures ci-dessus racontent, prises ensemble :

- **le modèle est autosuffisant** — `Distribution` est une structure de trois champs lisibles,
  pas un `Measure` de Mathlib ; un étudiant peut construire une distribution à la main (nous
  l'avons fait avec `Dcoin`) et la manipuler ;
- **la chaîne de dépendances est celle du cours** — vocabulaire (`Data`) → échantillon (`Sample`)
  → concentration (`MGF`/`BernoulliMGF`/`Hoeffding`) → union bound (`UnionBound`) → borne finie
  (`ERM`, `PacFiniteBound`) → agnostique (`Agnostic`), et en parallèle la branche géométrique du
  perceptron (`Perceptron/*`, `Convergence`, `Tightness`) ;
- **chaque constante chiffrée d'un manuel a son théorème** — `pac_finite_class_bound` porte le
  `log |H| + log(1/δ)` de la complexité d'échantillon, `chernoff_ineq` l'exponentielle de
  Hoeffding, et le théorème de convergence du perceptron sa borne en `1/γ²` ;
- **le serrage n'est pas un détail** — `Tightness` prouve que la borne perceptron n'est pas
  pessimiste : le contre-exemple `witnessPts`/`witnessLbl` fait exactement le nombre d'erreurs
  de la borne. C'est la différence entre « la preuve passe » et « la preuve dit quelque chose ».

C'est la promesse du compagnon natif : la visibilité du lake passe par le compilateur, pas par une
transcription.

**Trois leçons transversales** :

1. **La restriction au cas discret n'est pas une limitation, c'est une stratégie**. En évitant
   `Measure`/`ENNReal`, le lake reste *lisible* par un étudiant. Le coût : on ne peut pas
   raisonner sur des distributions continues. Le gain : on peut *vérifier* chaque théorème en
   quelques `#check`.
2. **La stratification est visible**. Le lake expose 14 modules, chacun avec une *responsabilité*
   claire (`Data` = modèle, `Sample` = tirage, `Hoeffding` = concentration, etc.). C'est le
   *même* découpage qu'un cours de machine learning classique (Valiant → Hoeffding → ERM →
   agnostique → perceptron).
3. **Le serrage distingue une borne d'une borne utile**. Une borne sans serrage est un majorant ;
   une borne avec serrage est *la* borne. La construction explicite de témoins
   (`witnessPts`/`witnessLbl`) est la signature d'un travail *complet*.

**Pour aller plus loin** :

- Visiter `MGF` et `BernoulliMGF` (cœur calculatoire de la concentration) — le README du lake
  donne la carte.
- Tester les bornes sur un cas concret : `Dcoin` + classe des seuils (`h(x) = x > k` pour `k` réel).
- Comparer avec le compagnon [2.8b-Theorie-PAC-Lean.ipynb](../../ML/DataScienceWithAgents/02-ML-Cours/2.8b-Theorie-PAC-Lean.ipynb)
  (côté série ML).

## Exercices

Les exercices suivants sont à compléter. Ils utilisent `Dcoin` (section 1) et les théorèmes
`#check`-és ci-dessus. Remplacer chaque `sorry` par une preuve ; les indices sont dans les
commentaires.

**Conventions C.1** : les cellules contiennent des commentaires `-- Exercice N : a completer`
et `-- TODO etudiant`, et des corps partiels (`theorem ... := by sorry`). Elles sont
syntaxiquement correctes et compilent (avec un warning `sorry`). Remplacer le `sorry` par une
preuve complète ; les indices sont en commentaires.

**Indications** : chaque exercice a un *indice* dans son titre/section md — utilisez-le pour
découvrir le théorème à appliquer.

**Barème indicatif** : 5-10 minutes par exercice (les preuves sont des *spécialisations*
directes des théorèmes du lake).

---

**Exercice 1** : erreur nulle contre soi-même. Prouver que l'hypothèse constante vraie fait une
erreur vraie nulle contre elle-même, sous `Dcoin`. Indice : `PacLearning.trueError_self`.

**Exercice 2** : la masse des échantillons vaut un. Prouver que pour des échantillons de taille
1 sur `Fin 2`, la somme des poids vaut 1. Indice : `PacLearning.sampleWeight_sum_one`.

**Exercice 3** : symétrie du désaccord. Prouver que l'erreur de `h` contre `f` égale celle de
`f` contre `h`. Indice : `PacLearning.trueError_comm`.

**Coût total** : < 1 minute par exercice (les preuves sont des `exact?` ou `apply` + `simp`).

### Exercice 1 : erreur nulle contre soi-même

**Pourquoi cet exercice est *fondamental*** :

L'égalité `trueError_self` est l'**une des 4 propriétés de base** de `trueError` (cf. Section 1).
Elle dit qu'une hypothèse `h` fait toujours zéro erreur contre elle-même. C'est une conséquence
*immédiate* de la définition : si `h = f`, alors pour toute instance `x`, `h(x) = f(x)`, donc la
condition de mal-classement est *toujours* fausse.

**Le théorème à appliquer** :

`PacLearning.trueError_self : ∀ {X : Type*} [Fintype X] (D : PacLearning.Distribution X)
(h : PacLearning.Hypothesis X), PacLearning.trueError D h h = 0`

**La spécialisation demandée** :

`PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0`

C'est exactement `trueError_self` appliqué à `D := Dcoin` et `h := (fun _ => true)` (la constante
vraie sur `Fin 2`).

**Le protocole de preuve** :

```lean
theorem exo1_self_zero :
    PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0 := by
  exact PacLearning.trueError_self Dcoin (fun _ => true)
```

**Sortie attendue** : `theorem exo1_self_zero : PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0` —
signature propre, sans `sorry`.

**Coût** : < 0.1 seconde (un seul `exact` après le `by`).

In [10]:
-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_self_zero :
    PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0 := by
  sorry

-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_self_zero :
        ──────────────▶ 🟨 declaration uses `sorry`
    PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0 := by
  sorry
--% env 9
--% prove 0

Raw input:
{"cmd": "-- Exercice 1 : a completer\n-- TODO etudiant\ntheorem exo1_self_zero :\n    PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0 := by\n  sorry", "env": 8}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 5, "column": 2},
   "goal": "⊢ (PacLearning.trueError Dcoin (fun x => true) fun x => true) = 0",
   "endPos": {"line": 5, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 22},
   "data": "declaration uses `sorry`"}],
 "env": 9}

### Exercice 2 : la masse des échantillons vaut un

**Pourquoi cet exercice** :

Le théorème `sampleWeight_sum_one` est la *pierre d'angle* de la théorie PAC (cf. Section 2). Il
dit que les échantillons forment eux-mêmes une distribution sur l'espace des fonctions
`Fin n → X`. Sans ce résultat, on ne pourrait pas raisonner sur « la probabilité qu'un tirage
d'échantillon aléatoire ait telle propriété ».

**Le théorème à appliquer** :

`PacLearning.sampleWeight_sum_one : ∀ {X : Type*} [Fintype X] (D : PacLearning.Distribution X)
(n : ℕ), ∑ S : Fin n → X, PacLearning.sampleWeight D S = 1`

**La spécialisation demandée** :

`∑ S : Fin 1 → Fin 2, PacLearning.sampleWeight Dcoin S = 1`

Pour `n = 1` sur `Fin 2`, les échantillons sont les fonctions `Fin 1 → Fin 2`, c'est-à-dire les
constantes : `fun _ => 0` (poids `1/2`) et `fun _ => 1` (poids `1/2`). La somme vaut `1`.

**Le protocole de preuve** :

```lean
theorem exo2_masse_un :
    ∑ S : Fin 1 → Fin 2, PacLearning.sampleWeight Dcoin S = 1 := by
  exact PacLearning.sampleWeight_sum_one Dcoin 1
```

**Sortie attendue** : `theorem exo2_masse_un : ∑ S : Fin 1 → Fin 2, PacLearning.sampleWeight Dcoin S = 1` —
signature propre, sans `sorry`.

**Coût** : < 0.1 seconde (un seul `exact`).

In [11]:
-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_masse_un :
    ∑ S : Fin 1 → Fin 2, PacLearning.sampleWeight Dcoin S = 1 := by
  sorry

-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_masse_un :
        ─────────────▶ 🟨 declaration uses `sorry`
    ∑ S : Fin 1 → Fin 2, PacLearning.sampleWeight Dcoin S = 1 := by
  sorry
--% env 10
--% prove 1

Raw input:
{"cmd": "-- Exercice 2 : a completer\n-- TODO etudiant\ntheorem exo2_masse_un :\n    \u2211 S : Fin 1 \u2192 Fin 2, PacLearning.sampleWeight Dcoin S = 1 := by\n  sorry", "env": 9}
Raw output:
{"sorries":
 [{"proofState": 1,
   "pos": {"line": 5, "column": 2},
   "goal": "⊢ ∑ S, PacLearning.sampleWeight Dcoin S = 1",
   "endPos": {"line": 5, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 21},
   "data": "declaration uses `sorry`"}],
 "env": 10}

### Exercice 3 : symétrie du désaccord

**Pourquoi cet exercice** :

Le théorème `trueError_comm` est l'**une des 4 propriétés de base** de `trueError` (cf. Section 1).
Il dit que le désaccord entre deux hypothèses est *symétrique* : `h` se trompe contre `f` autant
que `f` se trompe contre `h`. C'est une conséquence *immédiate* de la définition : la condition
de mal-classement est `h(x) ≠ f(x)`, qui est symétrique en `h` et `f`.

**Le théorème à appliquer** :

`PacLearning.trueError_comm : ∀ {X : Type*} [Fintype X] (D : PacLearning.Distribution X)
(h f : PacLearning.Hypothesis X), PacLearning.trueError D h f = PacLearning.trueError D f h`

**La spécialisation demandée** :

Pour toutes hypothèses `f h : PacLearning.Hypothesis (Fin 2)`,
`PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f`

C'est exactement `trueError_comm` appliqué à `D := Dcoin`.

**Le protocole de preuve** :

```lean
theorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :
    PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f := by
  exact PacLearning.trueError_comm Dcoin f h
```

**Sortie attendue** : `theorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :
PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f` —
signature propre, sans `sorry`.

**Coût** : < 0.1 seconde (un seul `exact`).

**Le piège classique** : oublier que les arguments de `trueError_comm` sont *implicites* sur `X`
et `Fintype X`. Lean les résout automatiquement depuis `Dcoin : PacLearning.Distribution (Fin 2)`
(donc `X := Fin 2` et l'instance `Fintype (Fin 2)` est dérivable).

In [12]:
-- Exercice 3 : a completer
-- TODO etudiant
theorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :
    PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f := by
  sorry

-- Exercice 3 : a completer
-- TODO etudiant
theorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :
        ─────────────▶ 🟨 declaration uses `sorry`
    PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f := by
  sorry
--% env 11
--% prove 2

Raw input:
{"cmd": "-- Exercice 3 : a completer\n-- TODO etudiant\ntheorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :\n    PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f := by\n  sorry", "env": 10}
Raw output:
{"sorries":
 [{"proofState": 2,
   "pos": {"line": 5, "column": 2},
   "goal":
   "f h : PacLearning.Hypothesis (Fin 2)\n⊢ PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f",
   "endPos": {"line": 5, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 21},
   "data": "declaration uses `sorry`"}],
 "env": 11}

## Conclusion

Ce compagnon a parcouru **quatorze modules** du lake `learning_theory_lean` en exécutant leurs
déclarations dans le noyau Lean : côté théorie PAC, `Data`, `Sample`, `SampleExpect`,
`Concentration`, `Hoeffding`, `ERM`, `UniformConcentration`, `UnionBound`, `PacFiniteBound`,
`Agnostic` ; côté géométrie, `Perceptron.Data`, `Perceptron`, `Convergence`, `Tightness`. Seuls
`MGF` et `BernoulliMGF` — le cœur calculatoire de la dérivation de concentration — ne sont
visités qu'en prose. Avant cette série de `#check`, la quasi-totalité de ces modules n'était
citée par aucun notebook du dépôt : leur contenu formel existait pour le compilateur seul.

**Trois concepts clés à retenir** :

1. **Le modèle est *fini et discret*** : une distribution est une fonction `X → ℝ` avec `nonneg`
   et `sum_one`. C'est moins expressif que `Measure`, mais *vérifiable* par un étudiant.
2. **La chaîne PAC est *exactement* celle du cours** : vocabulaire → échantillon → concentration
   → union bound → borne finie → agnostique. Chaque module a une responsabilité claire.
3. **Le serrage n'est pas un détail** : `novikoff_bound_is_sharp` montre que la borne du
   perceptron est *atteignable*, pas juste un majorant.

**Pour aller plus loin** :

- [SL-1 — Logical Learning](SL-1-LogicalLearning.ipynb) : la présentation Python de la série,
  figures à l'appui ;
- [2.8b-Theorie-PAC-Lean.ipynb](../../ML/DataScienceWithAgents/02-ML-Cours/2.8b-Theorie-PAC-Lean.ipynb)
  : le premier compagnon du lake, côté série ML (modèle et échantillon) ;
- le [README du lake](../../ML/learning_theory_lean/README.md) : la carte complète des modules,
  `MGF` et `BernoulliMGF` inclus ;
- [SL-2 — Knowledge-Based Learning](SL-2-KnowledgeBasedLearning.ipynb) : la suite de la série,
  du côté connaissance.

**Références** : Mohri, Rostamizadeh & Talwalkar, *Foundations of Machine Learning* (2e éd.),
ch. 2-3 ; Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, ch. 21 (perceptron).

**Quatre idées forces pour la suite** :

1. **Le formel n'est pas l'ennemi du pédagogique** : restreindre le modèle (au cas discret) le
   rend *plus lisible*, pas moins. Un étudiant peut construire une distribution à la main et
   raisonner dessus.
2. **Le `#check` est un outil de navigation** : voir la signature d'un théorème (sans la preuve)
   donne *ce qu'il dit* et *où il s'applique*, sans le bruit des 200 lignes de tactiques.
3. **Le `#print axioms` est un garde-fou** : il distingue un théorème *prouvé* d'un théorème qui
   repose sur un axiome non-défini. Pour chaque `#check` critique, c'est une vérification de
   *qualité formelle*.
4. **Le serrage est la moitié du travail** : une borne sans serrage est un majorant (parfois
   très pessimiste). Le contre-exemple `witnessPts`/`witnessLbl` prouve que la borne perceptron
   est *atteignable* — c'est la *complétude* de la théorie.